# 春第 5 回｜測定とデータの来歴

歌志内市と三島村では，教材を配る児童数は同じでも，説明先の学校数が違います。何を一つと数えるかによって準備量がどう変わるかを，学校統計の元の四項目から確かめます。

最初に本文の例を一緒に実行し，後半では自分で条件を選んで約 20 分の実習に取り組みます。

## 準備

Colab では「ドライブにコピーを保存」で自分用のノートを作り，上から準備セルを実行します。CSV は公開 URL から読み込むため，アップロードは不要です。ローカルでは，実習 ZIP を展開してノートブックと CSV を同じフォルダーに置くと，手元のファイルを使えます。ローカルでは ZIP に同梱した `ssdse-a-2026-guide.pdf` を開きます。Colab では同じ資料の <a href="https://www.nstac.go.jp/files/kaisetsu-A-2026.pdf" target="_blank" rel="noopener noreferrer">公式解説 PDF</a> を別タブで開いて参照します。

以下の読み込みセルを実行します。出力の `1741` は，授業用 CSV に入った市区町村の行数です。児童数や学校数とは区別します。

In [ ]:
from pathlib import Path
import pandas as pd

file_name = "ssdse-school-2024.csv"
csv_path = Path("data") / file_name
if not csv_path.is_file():
    csv_path = Path("../data") / file_name
if not csv_path.is_file():
    csv_path = Path(file_name)
if not csv_path.is_file():
    csv_path = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_name

schools = pd.read_csv(csv_path)
len(schools)

## 本文と一緒に実行

各節の問いを読んでから，コードを上から順に実行します。児童と学校を混ぜて数えないよう，図の軸，表の列名，数値の単位を対応させます。

### 教材の冊数と学校への説明件数

歌志内市と三島村で，児童 1 人に教材 1 冊，学校 1 校に説明 1 件を用意します。左の児童数と右の学校数を読み，どちらの地域に何を多く用意するかを考えます。その差を元の学校種別の列まで戻って確かめます。

In [ ]:
# 提供した作図セル。そのまま実行します。
import matplotlib.pyplot as plt
import pandas as pd

opening_schools = schools
opening_match = (opening_schools["code"] == "R01227") | (opening_schools["code"] == "R46303")
opening_regions = opening_schools.loc[opening_match]
opening_pupils = opening_regions["elementary_pupils"] + opening_regions["integrated_early_pupils"]
opening_counts = opening_regions["elementary_schools"] + opening_regions["integrated_schools"]

opening_figure, (pupil_axis, school_axis) = plt.subplots(
    1, 2, sharey=True, layout="constrained"
)
region_labels = ["Utashinai", "Mishima"]
pupil_bars = pupil_axis.barh(region_labels, opening_pupils, color="#286b70")
school_bars = school_axis.barh(region_labels, opening_counts, color="#825039")
pupil_axis.bar_label(pupil_bars, padding=3)
school_axis.bar_label(school_bars, padding=3)
pupil_axis.set(xlim=(0, 42), xticks=[0, 10, 20, 30, 40], xlabel="Pupils")
school_axis.set(xlim=(0, 5), xticks=[0, 1, 2, 3, 4], xlabel="Schools")
pupil_axis.invert_yaxis()
pupil_axis.spines[["top", "right"]].set_visible(False)
school_axis.spines[["top", "right"]].set_visible(False)
plt.show()

### 市区町村の行と元の項目を確認する

授業用 CSV の 1 行は 1 市区町村で，学校 1 校や児童 1 人ではありません。先頭の 3 行で地域の識別コードと名前を読み，この後の集計で何を足す必要があるか考えます。

In [ ]:
import pandas as pd

lecture_schools = schools.copy()
region_columns = ["code", "prefecture", "municipality"]
print(len(schools))
schools[region_columns].head(3)

歌志内市の行では，小学校数が 0 校なのに対象の児童が 35 人います。公式解説 PDF の項目コードを使い，四つの列がそれぞれどの学校種の学校・児童を数えるかを確かめます。

| 項目コード | 数えるもの | 単位 |
| --- | --- | --- |
| E2101 | 小学校 | 校 |
| E2501 | 小学校の児童 | 人 |
| E3901 | 義務教育学校 | 校 |
| E3905 | 義務教育学校前期課程の児童 | 人 |

四つの値は 2024 年 5 月 1 日の学校基本調査に基づきます。

In [ ]:
school_columns = [
    "code", "prefecture", "municipality",
    "elementary_schools", "elementary_pupils",
    "integrated_schools", "integrated_early_pupils",
]
schools.loc[schools["code"] == "R01227", school_columns]

### 同じ二地域の児童と学校をそれぞれ足す

教材の冊数には小学校児童と義務教育学校前期課程児童の人数を足します。説明件数には小学校と義務教育学校の校数を足します。歌志内市と三島村で作った二つの合計を，冒頭の図の左右の棒と照合します。

In [ ]:
lecture_schools["early_pupils"] = lecture_schools["elementary_pupils"] + lecture_schools["integrated_early_pupils"]
comparison_regions = (lecture_schools["code"] == "R01227") | (lecture_schools["code"] == "R46303")
pupil_columns = ["municipality", "elementary_pupils", "integrated_early_pupils", "early_pupils"]
lecture_schools.loc[comparison_regions, pupil_columns]

In [ ]:
lecture_schools["school_count"] = lecture_schools["elementary_schools"] + lecture_schools["integrated_schools"]
same_pupils = lecture_schools.loc[comparison_regions]
comparison_columns = ["municipality", "early_pupils", "elementary_schools",
                      "integrated_schools", "school_count"]
same_pupils[comparison_columns]

### 学校あたりの平均と学校ごとの人数

次は白糠町で「対象の学校 1 校あたり何人か」を考えます。分子の 203 人は二つの学校種の児童数です。分母にはどの学校数の列を足す必要があるでしょうか。同じ範囲の学校数で割り，平均を求めます。

In [ ]:
shiranuka = lecture_schools.loc[lecture_schools["code"] == "R01668"]
shiranuka_columns = [
    "municipality", "elementary_schools", "elementary_pupils",
    "integrated_schools", "integrated_early_pupils", "early_pupils", "school_count",
]
shiranuka[shiranuka_columns]

In [ ]:
shiranuka_pupils_per_school = shiranuka["early_pupils"] / shiranuka["school_count"]
shiranuka_pupils_per_school

二つの図は，白糠町の義務教育学校 2 校に 195 人をどう分けるかについての仮想例です。どちらも合計 203 人・3 校で，学校あたりの平均も同じになります。実際の学校別人数をこの平均から決められるか，二つの内訳を比べて考えます。

In [ ]:
# 提供した作図セル。そのまま実行します。
import matplotlib.pyplot as plt

allocation_figure, (first_axis, second_axis) = plt.subplots(
    1, 2, sharex=True, sharey=True, layout="constrained"
)
school_labels = ["Elementary", "Integrated A", "Integrated B"]
school_colors = ["#825039", "#286b70", "#286b70"]
first_bars = first_axis.barh(school_labels, [8, 97, 98], color=school_colors)
second_bars = second_axis.barh(school_labels, [8, 15, 180], color=school_colors)
first_axis.bar_label(first_bars, padding=3)
second_axis.bar_label(second_bars, padding=3)
first_axis.set(title="Hypothetical allocation 1", xlim=(0, 204),
               xticks=[0, 50, 100, 150, 200], xlabel="Pupils")
second_axis.set(title="Hypothetical allocation 2", xlabel="Pupils")
first_axis.invert_yaxis()
first_axis.spines[["top", "right"]].set_visible(False)
second_axis.spines[["top", "right"]].set_visible(False)
plt.show()

### 県内の数量を足す

岐阜県と奈良県について，教材と学校への説明を県全体で何件準備するか考えます。まず県に含まれる自治体の行数を数え，次に児童数と学校数の列を別々に足します。`size()` が返す自治体数を，児童や学校の合計と取り違えないようにします。

In [ ]:
prefecture_groups = lecture_schools.groupby("prefecture")
prefecture_names = ["岐阜県", "奈良県"]
prefecture_rows = prefecture_groups.size()
prefecture_rows.loc[prefecture_names]

In [ ]:
prefecture_totals = prefecture_groups[["early_pupils", "school_count"]].sum()
prefecture_totals.loc[prefecture_names]

県の合計は教材の総冊数や説明の総件数に使えます。学校種ごとの県内訳を元の列から求め，さらに一校ごとの配布数を知るには，どの単位の表が必要か考えます。

In [ ]:
type_columns = ["elementary_pupils", "integrated_early_pupils",
                "elementary_schools", "integrated_schools"]
prefecture_by_type = prefecture_groups[type_columns].sum()
prefecture_by_type.loc[prefecture_names]

## Jupyter 実習（約 20 分）

北海道か鹿児島県で，対象の児童に教材を配る場合と，学校に説明の機会を設ける場合の一方を選びます。必要な列が児童数か学校数かを公式解説で確かめ，選んだ道県の準備量を求めます。

これは **2024 年 5 月 1 日の記録を使う練習**です。現在の発注・訪問計画には，最新の人数や連絡先などの確認が必要です。共通課題は約 20 分（6・8・6 分），末尾は任意の発展です。

### 1. 依頼を選び，資料を確かめる（6 分）

地域を **北海道／鹿児島県**から選び，次の依頼を一つ選びます。

- **A 教材**：地域内の小学校の児童と義務教育学校の前期課程の児童へ，1 人に 1 冊を用意する。
- **B 説明**：地域内の小学校と義務教育学校を対象に，学校ごとに 1 件の説明の機会を用意する。

B で数えるのは対象の学校です。学校数だけから説明会の開催場所や訪問回数までは決められません。対象には，特別支援学校などの別の学校種は含めません。

`prefecture` の引用符の中を変更して実行し，その道県の自治体数を確認します。

計算の前に公式解説 PDF を開きます。**3 ページ「学校に関するデータについて」**，または **7 ページ「E 教育」の項目表**から，選ぶ列に関わる根拠を一つ見つけてください。対象となる学校種・学年，単位，収録年次のいずれかで構いません。本文を写すだけでなく，PDF のどの記載を使ったかが分かるように控えます。

- 開いたページと項目コード：
- その記載から確かめたこと（自分の言葉で）：
- 自分の依頼で，この確認が必要な理由：


In [ ]:
prefecture = "北海道"
area = schools.loc[schools["prefecture"] == prefecture]
len(area)

選んだ依頼で，何を一つと数えるかを先に書いてください。A は児童 1 人につき教材 1 冊，B は学校 1 校につき説明 1 件です。

- 道県：
- 依頼 A／B：
- 求める数量の単位と理由：

### 2. 列を選んで計算する（8 分）

以下は選んだ地域の元の 4 列の合計です。`len(area)` の出力は自治体数であり，児童数や学校数とは異なります。

| 列 | 収録内容 | 原項目 |
|---|---|---|
| `elementary_pupils` | 小学校児童数（人） | E2501 |
| `integrated_early_pupils` | 義務教育学校前期課程児童数（人） | E3905 |
| `elementary_schools` | 小学校数（校） | E2101 |
| `integrated_schools` | 義務教育学校数（校） | E3901 |

先ほど PDF で見つけた項目コードを，下の集計に使う列と照合します。対象や単位を確かめた場合は足す列の選択に，収録年次を確かめた場合は報告する時点に使います。これらは学校所在地による集計です。

In [ ]:
area[["elementary_pupils", "integrated_early_pupils",
      "elementary_schools", "integrated_schools"]].sum()

公式解説で確認した対象と単位に合わせ，依頼に必要な 2 列を足す一行を次のセルへ書いてください。

```python
quantity = area["一つ目の列名"] + area["二つ目の列名"]
```

`quantity` の一つの値は，1 自治体分の数量です。続くセルではその値をすべて足して道県の合計を求めます。

<details><summary>列の選び方のヒント</summary>

A は児童数の 2 列，B は学校数の 2 列を足します。列の名前が似ていても，人と校は別の単位です。元の列を上書きする必要はありません。

</details>

In [ ]:
# ここに quantity を作る一行を書きます。

自治体ごとの `quantity` を作ったら，次の 2 行の先頭の `#` を外して実行し，道県全体の合計を求めます。出た数字が児童数から作った冊数か，学校数から作った説明件数かを確認してください。

In [ ]:
# total = quantity.sum()
# total

### 3. 数量の意味を報告する（6 分）

最初に選んだ依頼へ戻り，計算結果で何をいくつ準備できるかを報告してください。2024 年 5 月 1 日の，学校所在地で集計した記録であることも明記します。結果の報告例に合わせ，常体で書いて構いません。

- **見積り**：どの道県の，どの学校種の，いつの記録を使い，何をいくつ用意する計算になったか。
- **資料の根拠**：PDF で見つけた項目コードとページを添え，その記載を列の選択・答えの単位・報告する時点のどこに使ったかを書く。例えば「解説の何ページで何を確認したため，この列を足した」のようにつなぐ。
- **次に必要な表**：実際に学校別に配送する，または説明の予定を組むなら，追加で必要な情報を一つ挙げ，理由を書く。

「この地域に住む子ども全員」と「この地域の対象学校に在籍する児童」は同じ対象とは限りません。元データで測っている範囲に沿って報告してください。

**提出前**にカーネルを再起動して最初から順に実行し，計算結果と記入した説明が残ったノートを保存します。任意課題を空欄にしても構いません。

### 発展（任意）

選んだ道県の合計が同じでも，学校別の人数の分布は一つに決まりません。教材を学校別に箱詰めする場面を考え，次に集める表の「1 行」と「必要な列」を提案してください。個人の名簿が必要かどうかも，目的に照らして考えます。

別の道県やもう一方の依頼で計算しても構いません。その場合は地域の選択から再実行し，報告の対象と単位も更新してください。

### 出典

統計センター [SSDSE-A-2026](https://www.nstac.go.jp/use/literacy/ssdse/) を加工。学校の値は 2024 年，地域区分は 2025 年 3 月 31 日時点。2026 は版の名前です。原統計と項目定義は同梱の公式解説，教材の列との対応は `ssdse-sources.md` にあります。